In [1]:
import fitz
import re 
import json
import nltk
nltk.download('punkt')
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
import pandas as pd 
from sqlalchemy import create_engine
import openpyxl

KeyboardInterrupt: 

In [22]:

LIFECYCLE_DESCRIPTIONS = {
    "Plan": "planning strategy risk assessment objectives policies governance preparation",
    "Do": "implementation execution security controls operational processes deployment",
    "Check": "audit monitoring review evaluation performance compliance verification",
    "Act": "improvement corrective actions optimization continuous improvement"
}
vectorizer = TfidfVectorizer()
phase_names = list(LIFECYCLE_DESCRIPTIONS.keys())
phase_texts = list(LIFECYCLE_DESCRIPTIONS.values())

phase_vectors = vectorizer.fit_transform(phase_texts)

#phase_vectors

def extract_lifecycle(text):
    text_vector = vectorizer.transform([text])
    similiarities = cosine_similarity(text_vector,phase_vectors)[0]
    best_match = phase_names[similiarities.argmax()]
    return best_match

In [23]:
## Normative references

def detect_normative(text):
    text_lower = text.lower()

    if 'shall' in text_lower:
        return True, 'shall'
    if 'may' in text_lower:
        return True, 'may'
    if 'should' in text_lower:
        return True, 'should'
    return False, None
    

In [24]:
## Keyword Extraction


def extract_keywords(text, max_keywords=5):

    words = re.findall(r'\b[a-zA-Z]{6,}\b', text.lower())

    freq = {}

    for w in words:
        freq[w] = freq.get(w, 0) + 1

    sorted_words = sorted(freq.items(), key=lambda x: x[1], reverse=True)

    return [w[0] for w in sorted_words[:max_keywords]]


In [25]:
## Create summary 
def create_summary(text, max_sentences=2):

    try:
        sentences = nltk.sent_tokenize(text)
    except LookupError:
        # Newer NLTK versions may require punkt_tab in addition to punkt.
        nltk.download('punkt', quiet=True)
        nltk.download('punkt_tab', quiet=True)
        sentences = nltk.sent_tokenize(text)

    return " ".join(sentences[:max_sentences])

In [ ]:
import fitz
import re
import json
import pandas as pd

PDF_PATH = r"D:\exercises\Thesis\bsi-standard-2001.pdf"
JSON_OUTPUT_PATH = r"D:\exercises\Thesis\bsi_standard.json"

BOX_CHARS = "┌┐└┘├┤┬┴┼│─═║╔╗╚╝╠╣╦╩╬█▄▀"

# -----------------------------
# HELPERS
# -----------------------------
def load_pdf_text(pdf_path):
    doc = fitz.open(pdf_path)
    try:
        return "".join("\n" + page.get_text() for page in doc)
    finally:
        doc.close()


def clean_content(text: str) -> str:
    text = re.sub(r'[^\x00-\x7F]+', ' ', text)

    lines = text.splitlines()
    cleaned_lines = []

    for line in lines:
        stripped = line.strip()

        if not stripped:
            continue

        # Remove noise
        if re.fullmatch(r"Page\s+\d+", stripped, flags=re.IGNORECASE):
            continue
        if re.search(r"Table of contents", stripped, flags=re.IGNORECASE):
            continue
        if re.search(r"Federal Office for Information Security", stripped, flags=re.IGNORECASE):
            continue
        if re.search(r"Copyright", stripped, flags=re.IGNORECASE):
            continue
        if re.search(r"BSI Standard 200-1", stripped, flags=re.IGNORECASE):
            continue
        if re.fullmatch(r"\d+", stripped):
            continue
        if any(ch in BOX_CHARS for ch in stripped):
            continue

        cleaned_lines.append(stripped)

    cleaned = " ".join(cleaned_lines)
    cleaned = re.sub(r"[\x00-\x1f\x7f]", " ", cleaned)
    cleaned = re.sub(r"(\b[\w]{2,})-\s+([\w]{2,}\b)", r"\1\2", cleaned)
    cleaned = re.sub(r"\s+", " ", cleaned).strip()

    return cleaned


def is_noise_title(title):
    t = title.lower()
    return any(x in t for x in [
        "introduction",
        "overview",
        "document",
        "structure",
        "appendix"
    ])


def split_into_chunks(text, max_sentences=2):
    sentences = re.split(r'(?<=[.!?]) +', text)

    chunks = []
    for i in range(0, len(sentences), max_sentences):
        chunk = " ".join(sentences[i:i + max_sentences]).strip()

        if len(chunk) > 50:
            chunks.append(chunk)

    return chunks


def is_reference_section(section_number: str, title: str) -> bool:
    title_lower = title.lower().strip()
    return (
        section_number.startswith("11.1")
        or "reference" in title_lower
    )


def extract_bsi_chunks(pdf_path):
    full_text = load_pdf_text(pdf_path)
    heading_pattern = re.compile(r"\n((\d+(?:\.\d+)*)\s+([A-Z][^\n]+))")
    matches = list(heading_pattern.finditer(full_text))

    chunks_data = []
    seen_chunks = set()

    for i, match in enumerate(matches):
        section_number = match.group(2)
        title = match.group(3).strip()

        # Skip unwanted sections
        if is_reference_section(section_number, title):
            continue
        if section_number.isdigit() and int(section_number) > 20:
            continue
        if re.fullmatch(r"V\s+\d+(?:\.\d+)*", title):
            continue
        if is_noise_title(title):
            continue

        start = match.end()
        end = matches[i + 1].start() if i + 1 < len(matches) else len(full_text)

        raw_content = full_text[start:end]
        content = clean_content(raw_content)

        if not content:
            continue

        # IMPORTANT: detect normative BEFORE chunking
        normative_flag, norm_type = detect_normative(content)

        # Skip weak sections early
        if norm_type not in ["shall", "should"]:
            continue

        lifecycle = extract_lifecycle(title + " " + content)
        keywords = extract_keywords(content)
        summary = create_summary(content)
        chunks = split_into_chunks(content)

        for idx, chunk in enumerate(chunks):
            if chunk in seen_chunks:
                continue
            seen_chunks.add(chunk)

            chunks_data.append({
                "source_standard": "BSI",
                "source_id": f"BSI-{section_number}",
                "section_number": section_number,
                "title": title,
                "parent": section_number.rsplit(".", 1)[0] if "." in section_number else None,
                "chunk_id": f"{section_number}_{idx}",
                "content": chunk,
                "summary": summary,
                "keywords": keywords,
                "lifecycle_phase": lifecycle,
                "normative": normative_flag,
                "normative_type": norm_type
            })

    return chunks_data


def save_chunks_as_json(chunks_data, output_path):
    with open(output_path, "w", encoding="utf-8") as f:
        json.dump(chunks_data, f, ensure_ascii=False, indent=2)


# -----------------------------
# MAIN EXTRACTION
# -----------------------------
records = extract_bsi_chunks(PDF_PATH)
save_chunks_as_json(records, JSON_OUTPUT_PATH)
df = pd.DataFrame(records)

print(f"Extracted {len(records)} clean chunks")

Extracted 112 clean chunks


## Transfer to EXCEL


In [28]:
df.to_excel('BSI_Standard.xlsx', index= False)

## Transfer to PostgreSQL

In [29]:
# PostgreSQL transfer
from sqlalchemy import create_engine
from urllib.parse import quote_plus

# Update these credentials for your environment.
db_user = "postgres"
db_password = "Ruban@1997#"
db_host = "localhost"
db_port = "5432"
db_name = "postgres"
table_name = "bsi_standard"

connection_url = (
    f"postgresql+psycopg2://{quote_plus(db_user)}:{quote_plus(db_password)}"
    f"@{db_host}:{db_port}/{db_name}"
)

engine = create_engine(connection_url)

# Writes DataFrame into PostgreSQL; replace table if it already exists.
df.to_sql(table_name, engine, if_exists="replace", index=False)

print(f"Loaded {len(df)} rows into {db_name}.{table_name}")

Loaded 112 rows into postgres.bsi_standard


## Transfer to Supabase


In [38]:
import requests
import json

SUPABASE_URL = "https://kaueqrfosfzpdhtnwdko.supabase.co"
SUPABASE_KEY = "sb_publishable_xb-nsEG892PB_8CgyulR0w_T1MVv2Z9"

url = f"{SUPABASE_URL}/rest/v1/bsi_standard"

headers = {
    "apikey": SUPABASE_KEY,
    "Authorization": f"Bearer {SUPABASE_KEY}",
    "Content-Type": "application/json",
    "Prefer": "return=minimal"
}

with open(r"D:\exercises\Thesis\bsi_standard.json", "r", encoding="utf-8") as f:
    data = json.load(f)

rows = []

for i, item in enumerate(data):

    standard = item.get("standard", {})

    rows.append({
    "chunk_id": item.get("chunk_id"),

    "standard_name": item.get("source_standard"),  
    "version": "1.0",                              
    "year": 2017,                                 
    "type": "governance",                          

    "source_id": item.get("source_id"),
    "section_number": item.get("section_number"),
    "title": item.get("title"),
    "parent": item.get("parent"),

    "lifecycle_phase": item.get("lifecycle_phase"),
    "normative": item.get("normative"),
    "normative_type": item.get("normative_type"),

    "keywords": item.get("keywords") or [],
    "summary": item.get("summary"),
    "content": item.get("content"),
})

# Insert
response = requests.post(url, headers=headers, json=rows)

print(response.status_code)
print(response.text)

201

